Libraries imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

Read and transformations

In [0]:
df_sales = spark.table("abinbev_case_silver.fact_sales")

df_top3_tradegroup = (
    df_sales
    .groupBy("btlr_org_lvl_c_desc", "trade_group_desc")
    .agg(F.sum("volume").alias("total_volume"))
)

window_region = Window.partitionBy("btlr_org_lvl_c_desc").orderBy(F.desc("total_volume"))

df_top3_tradegroup_ranked = (
    df_top3_tradegroup
    .withColumn("rank", F.dense_rank().over(window_region))
    .filter(F.col("rank") <= 3)
    .orderBy("btlr_org_lvl_c_desc", "rank")
)

Save as gold delta table

In [0]:
df_top3_tradegroup_ranked.write.mode("overwrite").format("delta").saveAsTable(
    "abinbev_case_gold.v_top3_tradegroup_region"
)

display(df_top3_tradegroup_ranked)